# 02 · LiDAR vs photogrammetry: do they give the same point cloud?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/bd-pointcloud/blob/main/notebooks/02_lidar_vs_photogrammetry.ipynb)

**Question:** if we scan the same object twice with Scaniverse, once in LiDAR mode and once in photogrammetry (photo) mode, how different are the two point clouds?

**Method**
1. Load both scans and make them comparable (same number of points, ground removed, same scale)
2. **Align** them, because the two scans are not in the same position or rotation (ICP)
3. **Measure** the difference: Chamfer distance, F-score, Hausdorff
4. Show *where* they differ with a colour heatmap

Background and literature are in `journal/01_scaniverse_lidar_vs_photogrammetry.md`.

**Runtime:** CPU is enough (no GPU needed). All the maths is in this notebook so we can read it.

---
## Settings

The only cell you normally need to edit. Put the two files you want to compare here.

Tip: also compare **two scans made with the same method** (e.g. `chair_lidar_01` vs `chair_lidar_02`). That tells us how much a method differs *from itself*, and a LiDAR-vs-photo difference only means something if it's bigger than that.

In [ ]:
SCAN_A = "chair_lidar_01.ply"   # reference, ideally the LiDAR scan
SCAN_B = "chair_photo_01.ply"   # the one we compare against it

UP_AXIS = "y"              # iPhone / ARKit / Scaniverse scans are Y-up
REMOVE_GROUND = True       # drop the floor, otherwise the floor dominates the comparison
GROUND_THRESHOLD_M = 0.02  # points within 2 cm of the floor plane count as floor
N_POINTS = 20_000          # points per cloud used for the comparison

## Get our project code

Colab only opened this *notebook*. Our helper code lives in the repo's `src/` folder, so we clone the repo into the Colab machine.

- **Public repo:** nothing to do.
- **Private repo:** create a GitHub token (GitHub → Settings → Developer settings → Fine-grained token, read-only access to this repo). In Colab, click the 🔑 **Secrets** icon in the left sidebar, add it as `GITHUB_TOKEN`, and turn on *Notebook access*.

> 🧠 **Python note:** `sys.path` is the list of folders Python searches when you write `import something`. Adding `src/` to it is what lets us `import pc_io` below.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/YOUR-GITHUB-USERNAME/bd-pointcloud.git"
REPO_DIR = "/content/bd-pointcloud"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"])  # already cloned: just update
else:
    url = REPO_URL
    try:
        from google.colab import userdata
        url = REPO_URL.replace("https://", "https://" + userdata.get("GITHUB_TOKEN") + "@")
        print("Using GITHUB_TOKEN from Colab secrets.")
    except Exception:
        print("No GITHUB_TOKEN secret, cloning as a public repo.")
    if subprocess.run(["git", "clone", "--depth", "1", url, REPO_DIR]).returncode != 0:
        raise RuntimeError("git clone failed: is the repo private? See the note above about GITHUB_TOKEN.")

sys.path.insert(0, f"{REPO_DIR}/src")
print("Code is in", REPO_DIR)

In [ ]:
!pip -q install plyfile trimesh

## Get the scans

Two options. Pick one by setting `USE_DRIVE` in the cell below.

**A. Google Drive (recommended for the team).** Keep one shared folder `bd-pointcloud/scans` on Drive. Each of us right-clicks the shared folder → *Organize* → *Add shortcut* → *My Drive*, so the path is the same for all three of us. Files stay there between Colab sessions.

**B. Upload from your computer.** Quick, but the files disappear when the Colab session ends.

Name the files `<object>_<method>_<take>.ply`, e.g. `rickshaw_lidar_01.ply`. See `data/README.md` in the repo.

Exported **OBJ**? Keep its `.mtl` and texture image (`.jpg`/`.png`) in the same folder, otherwise the colour is lost.

In [ ]:
import glob

USE_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/bd-pointcloud"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SCANS_DIR = f"{DRIVE_ROOT}/scans"
    RESULTS_DIR = f"{DRIVE_ROOT}/results"
else:
    from google.colab import files
    SCANS_DIR, RESULTS_DIR = "/content/scans", "/content/results"
    os.makedirs(SCANS_DIR, exist_ok=True)
    for name, data in files.upload().items():  # opens a file picker
        with open(os.path.join(SCANS_DIR, name), "wb") as fh:
            fh.write(data)

os.makedirs(RESULTS_DIR, exist_ok=True)
EXTS = (".ply", ".obj", ".glb", ".gltf", ".off", ".stl")
SCANS = sorted(p for p in glob.glob(f"{SCANS_DIR}/*") if p.lower().endswith(EXTS))
print(f"{len(SCANS)} scan(s) in {SCANS_DIR}")
for p in SCANS:
    print("  ", os.path.basename(p))

---
# Part 1 · Load and clean

## Load both scans

`load_points()` returns a numpy array `xyz` with one row per point (x, y, z), and `rgb` colours (or `None`).

**Check the sizes:** both are Scaniverse scans in metres, so the bounding boxes should be about the same and about the real object size. If one is 100× bigger, it's in different units.

In [ ]:
import numpy as np
from pc_io import load_points

def load(name):
    xyz, rgb, info = load_points(os.path.join(SCANS_DIR, name))
    size = xyz.max(axis=0) - xyz.min(axis=0)
    print(f"{name}: {info['kind']}, {len(xyz):,} points, size {size.round(2)} m")
    return xyz, rgb

xyz_a, rgb_a = load(SCAN_A)
xyz_b, rgb_b = load(SCAN_B)

## Make the point counts similar

Scans can have hundreds of thousands of points. Random subsampling to 50k keeps the next steps fast, and comparing clouds of similar density is fairer.

> 🧠 `rng.choice(n, k, replace=False)` picks `k` different random indices out of `0..n-1`. Indexing `xyz[idx]` then keeps just those rows.

In [ ]:
rng = np.random.default_rng(0)

def subsample(xyz, rgb, k):
    if len(xyz) <= k:
        return xyz, rgb
    idx = rng.choice(len(xyz), k, replace=False)
    return xyz[idx], (rgb[idx] if rgb is not None else None)

xyz_a, rgb_a = subsample(xyz_a, rgb_a, 50_000)
xyz_b, rgb_b = subsample(xyz_b, rgb_b, 50_000)
print(len(xyz_a), len(xyz_b))

## Remove the ground (RANSAC plane fit)

Scans include the floor or road under the object. We find the biggest **flat, horizontal** surface near the bottom and delete the points on it.

**RANSAC** in plain words: pick 3 random points → they define a plane → count how many points lie within 2 cm of it. Repeat a few hundred times and keep the plane with the most points.

*(Even better: crop the scan in the Scaniverse app before exporting.)*

In [ ]:
AXIS = {"x": 0, "y": 1, "z": 2}

def remove_ground(xyz, rgb, up_axis="y", threshold=0.02, iters=300, seed=0):
    rng = np.random.default_rng(seed)
    up = np.eye(3)[AXIS[up_axis]]
    heights = xyz[:, AXIS[up_axis]]
    low = xyz[heights <= np.percentile(heights, 30)]  # the floor is in the bottom part
    best = np.zeros(len(xyz), dtype=bool)
    for _ in range(iters):
        p0, p1, p2 = low[rng.choice(len(low), 3, replace=False)]
        normal = np.cross(p1 - p0, p2 - p0)
        if np.linalg.norm(normal) < 1e-9:
            continue                                    # the 3 points were on a line
        normal /= np.linalg.norm(normal)
        if abs(normal @ up) < 0.9:
            continue                                    # plane isn't horizontal, so not a floor
        on_plane = np.abs((xyz - p0) @ normal) < threshold
        if on_plane.sum() > best.sum():
            best = on_plane
    keep = ~best
    return xyz[keep], (rgb[keep] if rgb is not None else None), int(best.sum())

if REMOVE_GROUND:
    xyz_a, rgb_a, n = remove_ground(xyz_a, rgb_a, UP_AXIS, GROUND_THRESHOLD_M); print(f"A: removed {n:,} floor points")
    xyz_b, rgb_b, n = remove_ground(xyz_b, rgb_b, UP_AXIS, GROUND_THRESHOLD_M); print(f"B: removed {n:,} floor points")

xyz_a, rgb_a = subsample(xyz_a, rgb_a, N_POINTS)
xyz_b, rgb_b = subsample(xyz_b, rgb_b, N_POINTS)

## Look at them

Three flat views of each cloud: front (X-Y), top (X-Z), side (Y-Z). Check that the floor is gone and that it's the same object.

In [ ]:
import matplotlib.pyplot as plt

VIEWS = [(0, 1, "X-Y"), (0, 2, "X-Z"), (1, 2, "Y-Z")]

def show_views(clouds, titles, colors=None, size=0.5):
    fig, axes = plt.subplots(len(clouds), 3, figsize=(13, 4 * len(clouds)), squeeze=False)
    for row, (xyz, title) in enumerate(zip(clouds, titles)):
        c = colors[row] if colors is not None and colors[row] is not None else "tab:blue"
        for ax, (a, b, name) in zip(axes[row], VIEWS):
            ax.scatter(xyz[:, a], xyz[:, b], c=c, s=size)
            ax.set_title(f"{title}\n{name}", fontsize=9)
            ax.set_aspect("equal")
            ax.axis("off")
    plt.tight_layout()
    plt.show()

show_views([xyz_a, xyz_b], [SCAN_A, SCAN_B], colors=[rgb_a, rgb_b])

## Normalise: centre + unit sphere

Move each cloud's centre to (0, 0, 0) and scale it so the farthest point is at distance 1. **ModelNet40 and PointLLM normalise the same way**, so after this step we're comparing what a model would actually receive.

We keep `scale_a` (the radius in metres) so we can convert distances back to **cm** at the end.

In [ ]:
def normalise(xyz):
    centre = xyz.mean(axis=0)
    radius = np.max(np.linalg.norm(xyz - centre, axis=1))
    return (xyz - centre) / radius, radius

A, scale_a = normalise(xyz_a)
B, scale_b = normalise(xyz_b)
print(f"radius A = {scale_a:.3f} m, radius B = {scale_b:.3f} m, ratio = {scale_b / scale_a:.3f}  (close to 1.0 = same scale)")

---
# Part 2 · Align

The two scans were made separately, so one is rotated/shifted relative to the other. Before measuring, we line them up.

## Step 1: best rotation for matched pairs (Kabsch / SVD)

If we *knew* which point in A matches which point in B, the best rotation `R` and shift `t` have an exact formula using the SVD (singular value decomposition). This function is that formula.

In [ ]:
def best_fit_transform(src, dst):
    """R, t that minimise the distance between (src @ R.T + t) and dst, row by row."""
    cs, cd = src.mean(axis=0), dst.mean(axis=0)
    H = (src - cs).T @ (dst - cd)
    U, _, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:      # that was a mirror image, not a rotation, so fix it
        Vt[-1] *= -1
        R = Vt.T @ U.T
    t = cd - R @ cs
    return R, t

## Step 2: ICP (Iterative Closest Point)

We *don't* know the matches. ICP guesses them: pair every point with its **nearest neighbour** in the other cloud, apply `best_fit_transform`, and repeat until it stops improving.

> 🧠 `cKDTree` from SciPy is a data structure that finds nearest neighbours fast (instead of comparing every point with every other point).

In [ ]:
from scipy.spatial import cKDTree

def icp(src, dst, iters=50, tol=1e-6):
    tree = cKDTree(dst)
    cur = src.copy()
    prev_err = np.inf
    for _ in range(iters):
        dist, idx = tree.query(cur)                 # nearest neighbour of every point
        R, t = best_fit_transform(cur, dst[idx])
        cur = cur @ R.T + t
        err = dist.mean()
        if abs(prev_err - err) < tol:
            break
        prev_err = err
    return cur, err

## Step 3: try several starting angles

ICP only finds the *nearest* good fit, so if B starts facing the wrong way it can get stuck. Both scans have the same "up" (gravity), so we only need to try turning B around the up axis: every 30°, run ICP from each start, and keep the best.

In [ ]:
from scipy.spatial.transform import Rotation

def align(src, dst, up_axis="y", step_deg=30):
    best = (None, np.inf, None)
    for angle in range(0, 360, step_deg):
        R0 = Rotation.from_euler(up_axis, angle, degrees=True).as_matrix()
        moved, err = icp(src @ R0.T, dst)
        if err < best[1]:
            best = (moved, err, angle)
    return best

B_aligned, err, start_angle = align(B, A, UP_AXIS)
print(f"best start angle {start_angle}°, mean nearest-neighbour distance {err:.4f}")
show_views([np.vstack([A, B_aligned])], ["A (blue) + B (orange) aligned"],
           colors=[np.array(["tab:blue"] * len(A) + ["tab:orange"] * len(B_aligned))])

---
# Part 3 · Measure

For every point we ask: *how far is the nearest point in the other cloud?* That gives two lists of distances, A→B and B→A. From them:

| Metric | Meaning | Good is |
|---|---|---|
| **Chamfer** | average of the mean A→B and mean B→A distances | small |
| **Hausdorff** | the single worst distance (very sensitive to stray points) | small |
| **Hausdorff 95%** | 95th percentile instead of max, so less sensitive to noise | small |
| **Precision@τ** | % of B points within τ of A ("B didn't invent stuff") | high |
| **Recall@τ** | % of A points within τ of B ("B didn't miss stuff") | high |
| **F-score@τ** | combines precision and recall into one number (0–1) | high |

Distances are in **normalised units** (object radius = 1). τ = 0.02 is 1% of the object's diameter. We also print the cm equivalent using scan A's real size.

In [ ]:
def compare(A, B, thresholds=(0.01, 0.02, 0.05)):
    d_ab, _ = cKDTree(B).query(A)   # for each A point, distance to nearest B point
    d_ba, _ = cKDTree(A).query(B)
    m = {
        "chamfer": (d_ab.mean() + d_ba.mean()) / 2,
        "hausdorff": max(d_ab.max(), d_ba.max()),
        "hausdorff_95": max(np.percentile(d_ab, 95), np.percentile(d_ba, 95)),
    }
    for t in thresholds:
        precision = (d_ba < t).mean()
        recall = (d_ab < t).mean()
        m[f"precision@{t}"] = precision
        m[f"recall@{t}"] = recall
        m[f"fscore@{t}"] = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return m, d_ab, d_ba

metrics, d_ab, d_ba = compare(A, B_aligned)

to_cm = scale_a * 100
for k, v in metrics.items():
    extra = f"  (≈ {v * to_cm:.1f} cm)" if not k.startswith(("precision", "recall", "fscore")) else ""
    print(f"{k:>16}: {v:.4f}{extra}")

## Where do they differ?

Each point of A coloured by its distance to B. **Yellow = B is missing this part** (e.g. thin rods, spokes, shiny metal). Then the same the other way round.

In [ ]:
def heatmap(xyz, dist, title, vmax=0.05):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, (a, b, name) in zip(axes, VIEWS):
        sc = ax.scatter(xyz[:, a], xyz[:, b], c=dist, cmap="viridis", vmin=0, vmax=vmax, s=0.5)
        ax.set_title(f"{title}\n{name}", fontsize=9)
        ax.set_aspect("equal")
        ax.axis("off")
    fig.colorbar(sc, ax=axes, shrink=0.8, label="distance (normalised)")
    plt.show()

heatmap(A, d_ab, f"{SCAN_A}: distance to {SCAN_B}")
heatmap(B_aligned, d_ba, f"{SCAN_B}: distance to {SCAN_A}")

## Verdict and save

A **starting rule of thumb** (our own choice, not a standard, so adjust it once we've measured same-method repeatability):

- F-score@0.02 ≥ 0.8: **very similar**
- 0.5–0.8: **same overall shape, differences in detail**
- < 0.5: **clearly different**

The row is appended to `lidar_vs_photo.csv` in `RESULTS_DIR`, so all our comparisons end up in one table.

In [ ]:
import pandas as pd
from datetime import datetime

f = metrics["fscore@0.02"]
verdict = "very similar" if f >= 0.8 else "same shape, detail differs" if f >= 0.5 else "clearly different"
print(f"F-score@0.02 = {f:.3f} -> {verdict}")

row = {"date": f"{datetime.now():%Y-%m-%d %H:%M}", "scan_a": SCAN_A, "scan_b": SCAN_B,
       "radius_a_m": scale_a, "radius_b_m": scale_b, "verdict": verdict,
       **{k: round(float(v), 5) for k, v in metrics.items()}}
csv_path = f"{RESULTS_DIR}/lidar_vs_photo.csv"
pd.DataFrame([row]).to_csv(csv_path, mode="a", header=not os.path.exists(csv_path), index=False)
pd.read_csv(csv_path)